# Guardrails V2 + V3 walkthrough (D1 + D2 + D2b + D2c + D2d + D2e)

This notebook walks through the deterministic and LLM-audited guardrail layers added in **D1 (Guardrails V2)** and **D2 (Guardrails V3)**, plus the fixes and redesign that followed (**D2b**, **D2c**, **D2d**, **D2e**), cell by cell, using the same backend services that power the app. It picks up where [03_backend_query_answer_walkthrough.ipynb](03_backend_query_answer_walkthrough.ipynb) leaves off: that notebook covers extraction, RxNorm resolution, and retrieval; this one covers what happens to the *generated answer* after that — the coverage-driven contract, post-generation validation, and the per-citation faithfulness critic with bounded regeneration.

**D1 — Guardrails V2** (`src/query_answer/contract.py`, `src/query_answer/validation.py`):
- `build_answer_contract(understanding, coverage)` turns the deterministic coverage report into a list of `must_mention` / `must_caveat` `AnswerContractItem`s, plus an answer-level `coverage_level` (`direct`/`partial`/`limited`/`none`). Each intent item's `required_sections` is now the section(s) *actually matched* for that intent (`EvidenceCoverageItem.matched_sections`), not a static allow-list (D2b).
- The contract is fed into the synthesis prompt as `answer_contract` (see `EvidenceAnswerSynthesizer.build_evidence_packet`).
- `validate_and_enforce(answer, contract)` runs after synthesis: deterministically appends any `must_caveat` the model dropped, flags yes/no medical-advice framing, relocates any bullet with no citations into `limitations` as a caveat instead of leaving it in the answer body (D2d), and records unaddressed `must_mention` topics as `ValidationFinding`s.

**D2 — Guardrails V3, original shape (now superseded by D2e):** introduced a deterministic "support status" floor that ran on every bullet, with or without an LLM, classifying `strong`/`partial`/`limited`/`none` purely from whether a bullet's cited section *category* matched the topic it was about. An optional, off-by-default LLM critic could override that floor per bullet.

**D2b — fix cross-intent support-status leakage** and **D2c — fix evidence-packet truncation starving later label sections**: two bugs found by running this exact notebook against real queries, fixed in the floor and in `build_evidence_packet` respectively. Both are described in the "Update" notes near the end of this notebook, kept for the historical record.

**D2d — fix uncited "sources" and cross-claim gap leak** (`src/query_answer/validation.py`, `src/query_answer/critic.py`): an uncited bullet was showing up in the Sources list as a "No citation" badge — but an uncited bullet isn't a *source*, it's a *limitation*, so `validate_and_enforce` now relocates it there deterministically instead. The floor's "unresolved gap" check (downgrading `strong` to `partial`) was also scoped to the bullet's own topic instead of the whole contract, so an unrelated open gap elsewhere couldn't drag down an otherwise well-supported bullet.

**D2e — replace the deterministic floor with a source-faithfulness LLM critic** (`src/query_answer/critic.py`, `src/query_answer/models.py`) — **the current design, and what the rest of this notebook demonstrates:** the deterministic floor from D2/D2b/D2d measured something real but narrow — does a citation's section *category* match the topic it's about — and never read the cited text or the final response. That check was almost always trivially satisfied, since the same model call that writes a bullet also picks its citation. D2e deletes the floor entirely (`deterministic_support_status`, `apply_deterministic_statuses`, `_topic_required_sections`, `_addressed_intent_sections`, `_has_unresolved_gap` are all gone) and makes the LLM critic — now **on by default** (`enable_answer_critic: true`) — the *only* source of a support-status badge. The critic is also re-scoped from per-*bullet* to per-*citation* (`EvidenceCitation.support_status`, not `EvidenceBullet.support_status` — a bullet can carry more than one citation, and each is now judged independently), against a new two-axis, five-tier taxonomy:

- `accurate` — the cited label text supports the claim, and the response reflects it.
- `not_reflected` — the cited text supports the claim, but the response doesn't mention it.
- `contradicted` — the cited text supports the claim, but the response says something that conflicts with it.
- `misrepresented` — the cited text doesn't say what the claim says, and the response doesn't repeat the incorrect claim.
- `misrepresented_used` — the cited text doesn't say what the claim says, and the response repeats or relies on it anyway.

The critic prompt is deliberately minimal: just the query, the final response, the limitations, and — for each citation actually used — its claim text and the real retrieved label-section text for that citation (the same text, same D2c truncation, the synthesis model saw). It never receives the full evidence packet, `label_product_context`, or `answer_contract`. The bullet-level `topic` field (which only ever existed to feed the now-deleted floor) is gone too; the contract-level `topic` on `AnswerContractItem` is untouched and still drives `validate_and_enforce`'s deterministic caveat enforcement. The accepted trade-off: every live query now costs a second LLM call (synthesis + critic), and a third/fourth pair if regeneration triggers — and when the critic can't run (no LLM configured, or explicitly disabled, as in demo mode), citations simply carry no `support_status` and no badge renders, rather than falling back to a structural guess.

Section 14 collects gaps and edge cases worth a second look, including the ones D2e itself doesn't close.

## 0. Setup

Run this notebook from the project root or directly from `notebooks/`. All live-LLM toggles below default to `False`, so running the whole notebook top to bottom makes no paid API calls — every guardrail step is demonstrated with hand-built mock answers and mock critic responses first, with an optional live cell after each one.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repository root.")


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

print(f"Repository root: {ROOT}")

Repository root: /Users/Marie_Cordes/code/mariecordes/rx-ray


In [2]:
from src.dossier.builder import DossierBuilder
from src.dossier.openfda_store import OpenFDALabelStore
from src.dossier.rxnorm_store import RxNormParquetStore
from src.query_answer import critic
from src.query_answer.config import QueryAnswerParameters, load_query_answer_parameters
from src.query_answer.context import (
    build_context_targeted_evidence,
    merge_context_evidence_into_secondary,
    merge_context_evidence_into_understanding,
)
from src.query_answer.contract import build_answer_contract
from src.query_answer.coverage import build_evidence_coverage
from src.query_answer.critic import (
    apply_citation_statuses,
    critique_answer,
    finalize_answer_critique,
)
from src.query_answer.secondary import build_secondary_evidence
from src.query_answer.service import QueryAnswerService
from src.query_answer.synthesizer import EvidenceAnswerSynthesizer
from src.query_answer.validation import validate_and_enforce
from src.query_understanding.extractor import HybridQueryExtractor
from src.query_understanding.service import QueryUnderstandingService


def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value


def show_json(value):
    display(JSON(to_jsonable(value)))


def as_frame(rows):
    return pd.DataFrame([to_jsonable(row) for row in rows])

## 1. Choose a query and runtime toggles

This is the same multi-drug, multi-intent query used in the demo fixture and in [04_interaction_evidence_map_debug.ipynb](04_interaction_evidence_map_debug.ipynb): a current medication (cetirizine), two mentioned drugs (ibuprofen, aspirin) that trigger `interaction_check`, plus an allergy and a context phrase that trigger context-targeted lookups. It's a good stress test for the contract because it should produce both `must_mention` and `must_caveat` items across more than one intent.

Flip `RUN_LIVE_ANSWER_SYNTHESIS` / `RUN_LIVE_CRITIC` to `True` only when you want to spend real API calls against the configured models. Leave `ENABLE_ANSWER_CRITIC_OVERRIDE = None` to use whatever `conf/base/parameters.yml` says (`enable_answer_critic`, **on by default since D2e**); set it to `True`/`False` to force one path for this notebook run only.

In [3]:
QUERY = (
    # "I currently take cetirizine for my pollen allergy. can i take both "
    # "ibuprofen and aspirin against swollen eyes?"
    "Can I take ibuprofen for my migraine if I'm allergic to aspirin?"
)

RUN_LIVE_ANSWER_SYNTHESIS = True
RUN_LIVE_CRITIC = True
ENABLE_ANSWER_CRITIC_OVERRIDE: bool | None = None

base_parameters = load_query_answer_parameters()
if ENABLE_ANSWER_CRITIC_OVERRIDE is None:
    parameters = base_parameters
else:
    parameters = base_parameters.__class__(
        **{
            **base_parameters.__dict__,
            "enable_answer_critic": ENABLE_ANSWER_CRITIC_OVERRIDE,
        }
    )

print(f"Query: {QUERY}")
print("enable_answer_critic:", parameters.enable_answer_critic)
print("critic_max_regenerations:", parameters.critic_max_regenerations)
print(
    "Answer-synthesis LLM configured:",
    bool(os.getenv("ANSWER_SYNTHESIS_OPENAI_API_KEY") and os.getenv("ANSWER_SYNTHESIS_OPENAI_MODEL")),
)
print("Critic LLM falls back to the same env vars unless ANSWER_CRITIC_OPENAI_* is set.")

Query: Can I take ibuprofen for my migraine if I'm allergic to aspirin?
enable_answer_critic: True
critic_max_regenerations: 1
Answer-synthesis LLM configured: True
Critic LLM falls back to the same env vars unless ANSWER_CRITIC_OPENAI_* is set.


## 2. Instantiate the backend services

Same request-time stack as the FastAPI app: RxNorm parquet files plus OpenFDA live lookup with a local cache.

In [4]:
rxnorm_store = RxNormParquetStore()
openfda_store = OpenFDALabelStore(use_cache=True, allow_live=True)
builder = DossierBuilder(rxnorm_store=rxnorm_store, openfda_store=openfda_store)

extractor = HybridQueryExtractor()
understanding_service = QueryUnderstandingService(builder=builder, extractor=extractor)
synthesizer = EvidenceAnswerSynthesizer()
query_answer_service = QueryAnswerService(
    builder=builder,
    understanding_service=understanding_service,
    synthesizer=synthesizer,
)

print("Services ready.")

Services ready.


## 3. Run query understanding

Same object `POST /query-understanding` returns: extraction, RxNorm resolution, and the primary dossier. See notebook 03 for a deeper look at each step inside this call.

In [5]:
understanding = understanding_service.understand(
    QUERY,
    openfda_limit=parameters.default_openfda_limit,
)

print("Extraction mode:", understanding.extraction_mode)
print("Warnings:", understanding.warnings)
print("Errors:", understanding.errors)
show_json(understanding.state)

Extraction mode: hybrid
Warnings: []
Errors: []


<IPython.core.display.JSON object>

## 4. Build secondary and context-targeted evidence

This mirrors `QueryAnswerService.answer()`: secondary evidence for the other mentioned/current drugs, then context-targeted lookups for the allergy/condition/patient-context state, merged back into `understanding` and `secondary_evidence`. The deterministic coverage report and the contract both depend on having this evidence in hand first.

In [6]:
secondary_evidence = build_secondary_evidence(understanding, builder, parameters)
context_evidence = build_context_targeted_evidence(
    understanding, secondary_evidence, builder, parameters
)
understanding = merge_context_evidence_into_understanding(understanding, context_evidence)
secondary_evidence = merge_context_evidence_into_secondary(secondary_evidence, context_evidence)

display(
    as_frame(
        [
            {
                "mention_text": item.mention_text,
                "role": item.role,
                "resolved_name": item.resolved_concept.name,
                "labels_found": item.label_evidence.labels_found if item.label_evidence else 0,
                "interaction_labels_found": (
                    item.interaction_label_evidence.labels_found
                    if item.interaction_label_evidence
                    else 0
                ),
                "retrieval_modes": item.retrieval_modes,
            }
            for item in secondary_evidence
        ]
    )
)
display(
    as_frame(
        [
            {
                "target_label": item.target_label,
                "target_category": item.target_category,
                "resolved_name": item.resolved_concept.name,
                "searched_fields": item.searched_fields,
                "labels_found": item.label_evidence.labels_found if item.label_evidence else 0,
            }
            for item in context_evidence
        ]
    )
)

,mention_text,role,resolved_name,labels_found,interaction_labels_found,retrieval_modes
0,aspirin,allergy,aspirin,12,6,"[standard_secondary_label_lookup, cache, inter..."


,target_label,target_category,resolved_name,searched_fields,labels_found
0,migraine,condition,ibuprofen,"[indications_and_usage, warnings, contraindica...",3
1,migraine,condition,aspirin,"[indications_and_usage, warnings, contraindica...",3


## 5. Deterministic coverage report — `build_evidence_coverage`

This is the input to D1's contract builder: did retrieval actually find evidence for each extracted state item and intent? Each row's `status` is one of `addressed` / `not_found_in_evidence` / `not_retrieved` / `out_of_scope`.

In [7]:
coverage = build_evidence_coverage(
    understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
)

print("Summary counts:", coverage.summary_counts)
as_frame(coverage.items)

Summary counts: {'addressed': 6, 'not_found_in_evidence': 0, 'not_retrieved': 0, 'out_of_scope': 0}


,category,label,status,reason,matched_evidence,source_id,section,matched_sections,target_rxcui
0,primary_drug,ibuprofen,addressed,Drug label evidence was retrieved for ibuprofen.,NaN,NaN,NaN,[],5640
1,mentioned_drug,aspirin,addressed,Drug label evidence was retrieved for aspirin.,NaN,NaN,NaN,[],1191
2,allergy,aspirin,addressed,The retrieved label text explicitly mentions t...,"...patients who have experienced asthma, urtic...",5e595111-8477-4ec7-8d32-a0aeabc04e1a,contraindications,[],NaN
3,condition,migraine,addressed,The retrieved label text explicitly mentions t...,...right before or after heart surgery Ask a d...,460cceff-0b2e-cbfe-e063-6394a90aa46f,warnings,[],5640
4,intent,allergy_context_check,addressed,"Retrieved label sections (contraindications, w...",...CONTRAINDICATIONS Ibuprofen tablets are con...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,contraindications,"[contraindications, warnings]",5640
5,intent,interaction_check,addressed,Drug-interactions label text was retrieved for...,...Preexisting asthma Patients with asthma may...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,drug_interactions,[drug_interactions],5640


## 6. Build the answer contract — `build_answer_contract` (D1, new in this work)

Turns the coverage report into a checklist synthesis must satisfy: `must_mention` items name a topic with evidence available; `must_caveat` items are statements that must appear in `limitations`, verbatim or rephrased. Watch for the always-present `interaction_terminology` caveat whenever `interaction_check` is one of the extracted intents — it's emitted regardless of how good the interaction evidence is, which matters for how D2 classifies support status later (see section 10).

In [8]:
contract = build_answer_contract(understanding, coverage)

print("Coverage level:", contract.coverage_level)
as_frame(contract.items)

Coverage level: direct


,kind,topic,intent,statement,evidence_available,required_sections,coverage_category,coverage_label,target_rxcui
0,must_mention,allergy_context_check,allergy_context_check,Address what the retrieved allergy label secti...,True,"[contraindications, warnings]",intent,allergy_context_check,None
1,must_mention,interaction_check,interaction_check,Address what the retrieved drug interaction la...,True,[drug_interactions],intent,interaction_check,None
2,must_caveat,interaction_terminology,interaction_check,RxNorm terminology overlap is not evidence of ...,True,[],intent,interaction_check,None


## 7. Confirm the contract reaches the synthesis evidence packet

`EvidenceAnswerSynthesizer.build_evidence_packet` embeds the contract under `answer_contract`, alongside the non-citable `label_product_context` block (product/formulation info that must never satisfy a citation requirement). `label_sections` itself is bounded by `label_section_payloads()` (D2c): it deduplicates near-identical boilerplate repeated across a drug's many label records, caps each section to `MAX_SECTION_ENTRIES`, and guarantees any section the contract already relies on (`required_label_sections(contract)`) survives the overall `MAX_LABEL_SECTIONS` cap. The per-section breakdown below should always include `drug_interactions` for this query — before D2c it didn't, because ibuprofen alone has 17+ non-empty entries in earlier-priority sections.

In [9]:
evidence_packet = synthesizer.build_evidence_packet(
    understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    contract=contract,
)

print("Top-level keys:", list(evidence_packet.keys()))
print("label_sections:", len(evidence_packet.get("label_sections", [])))
print("label_product_context:", len(evidence_packet.get("label_product_context", [])))

section_counts = (
    pd.Series(
        [
            (entry["evidence_scope"], entry["drug_name"], entry["section"])
            for entry in evidence_packet.get("label_sections", [])
        ]
    )
    .value_counts()
    .rename("entries")
)
display(section_counts)
print(
    "drug_interactions present in label_sections?",
    any(
        entry["section"] == "drug_interactions"
        for entry in evidence_packet.get("label_sections", [])
    ),
)
show_json(evidence_packet["answer_contract"])

Top-level keys: ['query', 'state', 'resolved_primary_drug', 'label_evidence_scope', 'ingredient_fallback', 'rxnorm_relationship_summary', 'secondary_drug_evidence', 'context_targeted_evidence', 'label_sources', 'label_sections', 'label_product_context', 'answer_contract', 'retrieval_notes']
label_sections: 32
label_product_context: 12


(primary, None, warnings)                  3
(primary, None, pregnancy)                 3
(primary, None, indications_and_usage)     3
(secondary, aspirin, boxed_warning)        3
(secondary, aspirin, contraindications)    3
(secondary, aspirin, warnings)             3
(secondary, aspirin, drug_interactions)    3
(secondary, aspirin, pregnancy)            3
(primary, None, drug_interactions)         2
(primary, None, lactation)                 2
(primary, None, boxed_warning)             1
(primary, None, contraindications)         1
(primary, None, adverse_reactions)         1
(secondary, aspirin, lactation)            1
Name: entries, dtype: int64

drug_interactions present in label_sections? True


<IPython.core.display.JSON object>

## 8. Draft an answer to audit

To keep the rest of this notebook free of paid API calls by default, this cell builds a deliberately imperfect mock answer: it cites one real, allowed section, but skips at least one `must_caveat` from the contract so the next section has something real to enforce. Flip `RUN_LIVE_ANSWER_SYNTHESIS` in section 1 to use the real synthesizer call instead.

In [10]:
from src.query_answer.models import EvidenceAnswer, EvidenceBullet, EvidenceCitation

allowed_citations = synthesizer.allowed_citations(evidence_packet)
first_citation = next(iter(sorted(allowed_citations)), None)

mock_answer = EvidenceAnswer(
    response="The retrieved labels mention an interaction-style caution for these medications.",
    bullets=[
        EvidenceBullet(
            text="A retrieved label section mentions the other medication.",
            citations=(
                [EvidenceCitation(source_id=first_citation[0], section=first_citation[1])]
                if first_citation
                else []
            ),
        ),
    ],
    limitations=[],
    safety_note="This is an educational summary of retrieved public evidence, not medical advice.",
)

if RUN_LIVE_ANSWER_SYNTHESIS:
    synthesis_result = synthesizer.synthesize(
        QUERY,
        understanding,
        secondary_evidence=secondary_evidence,
        context_evidence=context_evidence,
        contract=contract,
    )
    print("Warnings:", synthesis_result.warnings)
    print("Errors:", synthesis_result.errors)
    draft_answer = synthesis_result.answer or mock_answer
else:
    print("Skipped live answer synthesis; using the hand-built mock answer.")
    draft_answer = mock_answer

show_json(draft_answer)

Warnings: []
Errors: []


<IPython.core.display.JSON object>

## 9. Validate and enforce the contract — `validate_and_enforce` (D1)

Runs after synthesis: appends any `must_caveat` missing from `limitations`, flags yes/no medical-advice framing, and records unaddressed `must_mention` topics as findings. Compare `limitations` before and after — anything new was added deterministically, not by the model.

In [11]:
validated_answer, validation = validate_and_enforce(draft_answer, contract)

print("Limitations before:", draft_answer.limitations)
print("Limitations after: ", validated_answer.limitations)
show_json(validation)

Limitations before: ['The retrieved labels do not provide enough information to determine what to do for an individual with an aspirin allergy; they only describe label warnings/contraindications for aspirin/NSAID allergic-type reactions.', 'The retrieved labels include ibuprofen–aspirin interaction information (e.g., interference with aspirin’s antiplatelet activity when both are taken), but this is not the same as proving an allergy cross-reaction for your specific situation.', 'RxNorm terminology overlap is not evidence of a clinical interaction.']
Limitations after:  ['The retrieved labels do not provide enough information to determine what to do for an individual with an aspirin allergy; they only describe label warnings/contraindications for aspirin/NSAID allergic-type reactions.', 'The retrieved labels include ibuprofen–aspirin interaction information (e.g., interference with aspirin’s antiplatelet activity when both are taken), but this is not the same as proving an allergy cro

<IPython.core.display.JSON object>

## 10. No more deterministic floor (removed in D2e)

Through D2/D2b/D2d, this section ran `apply_deterministic_statuses` here, unconditionally, before the critic. D2e deleted that function entirely (along with `_topic_required_sections`, `_addressed_intent_sections`, `_has_unresolved_gap`) — there is no structural floor anymore, and no fallback `support_status` is assigned at this point. The next section's critic is now the *only* place a citation gets a `support_status`, and it judges each citation against the real cited text rather than a section-category match.

## 11. The LLM critic — `critique_answer` (D2, redesigned in D2e)

The critic now reads a minimal, per-citation input — built by `critic._critic_input`, capped to `MAX_CRITIC_BULLETS` bullets and `MAX_CRITIC_CITATIONS` total citations — and judges each citation against the real cited label text and the final response, never the full evidence packet. It assigns one of the five `support_status` tiers per citation (see the intro cell), flags `issues` (`overconfident`/`invented_detail`), and may request regeneration via `needs_regeneration` (only when at least one citation is `contradicted`/`misrepresented`/`misrepresented_used`). The mock requester below stands in for an LLM response so this runs with no API calls; flip `RUN_LIVE_CRITIC` in section 1 to call the real model instead.

In [12]:
def mock_critic_requester(*, messages, prompt_config):
    """Stands in for the critic LLM call (tests use the same pattern)."""
    return {
        "citations": [
            {
                "bullet_index": 0,
                "citation_index": 0,
                "support_status": "not_reflected",
                "rationale": "Cited text supports the claim, but the response doesn't mention it.",
                "issues": [],
            }
        ],
        "global_findings": [],
        "needs_regeneration": False,
    }


if RUN_LIVE_CRITIC and critic._critic_llm_configured():
    critic_requester = None  # None => critique_answer makes a real API call
else:
    if RUN_LIVE_CRITIC:
        print("RUN_LIVE_CRITIC is True but no critic/synthesis LLM is configured; using the mock instead.")
    critic_requester = mock_critic_requester

outcome = critique_answer(
    query=QUERY,
    answer=validated_answer,
    evidence_packet=evidence_packet,
    json_requester=critic_requester,
)

print("needs_regeneration:", outcome.needs_regeneration)
show_json(outcome.critique)

critiqued_answer = apply_citation_statuses(validated_answer, outcome.critique)
display(
    as_frame(
        [
            {
                "bullet_index": bullet_index,
                "citation_index": citation_index,
                "text": bullet.text,
                "source_id": citation.source_id,
                "section": citation.section,
                "support_status": citation.support_status,
            }
            for bullet_index, bullet in enumerate(critiqued_answer.bullets)
            for citation_index, citation in enumerate(bullet.citations)
        ]
    )
)

needs_regeneration: False


<IPython.core.display.JSON object>

,bullet_index,citation_index,text,source_id,section,support_status
0,0,0,Ibuprofen labels list **contraindication** for...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,contraindications,accurate
1,1,0,Ibuprofen labels include an **allergy alert** ...,4bd1345d-d116-426e-825f-bbf29381746a,warnings,accurate
2,2,0,The ibuprofen label describes **cross reactivi...,5e595111-8477-4ec7-8d32-a0aeabc04e1a,drug_interactions,accurate
3,3,0,"For migraine-related use, the retrieved ibupro...",4bd1345d-d116-426e-825f-bbf29381746a,indications_and_usage,accurate


## 12. Full orchestration — `finalize_answer_critique` (D2 entrypoint, redesigned in D2e)

This is the function `QueryAnswerService.answer()` calls. Two runs below show both branches without spending real tokens:

**(a) critic disabled** — since D2e there's no deterministic floor to fall back to, so every citation simply carries `support_status=None` and `critique.enabled` is `False`.

**(b) critic enabled, forced regeneration** — a mock critic that says `needs_regeneration=True` on its first call (a `misrepresented` citation) and `False` on the recheck after regeneration (`accurate`), paired with a mock synthesis requester for the regeneration call. This is the same pattern `tests/test_critic.py::test_finalize_answer_critique_regenerates_exactly_once` uses to prove regeneration happens **exactly once**, never in a loop.

In [13]:
deterministic_only_parameters = parameters.__class__(
    **{**parameters.__dict__, "enable_answer_critic": False}
)

answer_a, critique_a, validation_a = finalize_answer_critique(
    query=QUERY,
    understanding=understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    answer=draft_answer,
    contract=contract,
    validation=validation,
    synthesizer=synthesizer,
    parameters=deterministic_only_parameters,
)

print("(a) critic.enabled:", critique_a.enabled, "| source:", critique_a.source)
[c.support_status for b in answer_a.bullets for c in b.citations]

(a) critic.enabled: False | source: none


[None, None, None, None]

In [14]:
regeneration_critic_calls = {"count": 0}


def forced_regeneration_critic(*, messages, prompt_config):
    regeneration_critic_calls["count"] += 1
    first_call = regeneration_critic_calls["count"] == 1
    return {
        "citations": [
            {
                "bullet_index": 0,
                "citation_index": 0,
                "support_status": "misrepresented" if first_call else "accurate",
                "rationale": "Forced for this walkthrough.",
                "issues": ["invented_detail"] if first_call else [],
            }
        ],
        "global_findings": [],
        "needs_regeneration": first_call,
    }


def mock_regeneration_synth_requester(*, messages, prompt_config):
    return {
        "response": "Regenerated for this walkthrough.",
        "bullets": (
            [
                {
                    "text": "Regenerated bullet citing the same allowed source.",
                    "citations": [
                        {"source_id": first_citation[0], "section": first_citation[1]}
                    ],
                }
            ]
            if first_citation
            else []
        ),
        "limitations": [],
    }


os.environ.setdefault("ANSWER_SYNTHESIS_OPENAI_API_KEY", "notebook-mock-key")
os.environ.setdefault("ANSWER_SYNTHESIS_OPENAI_MODEL", "notebook-mock-model")
mock_synthesizer = EvidenceAnswerSynthesizer(
    parameters=parameters,
    json_requester=mock_regeneration_synth_requester,
)

critic_enabled_parameters = parameters.__class__(
    **{**parameters.__dict__, "enable_answer_critic": True, "critic_max_regenerations": 1}
)

answer_b, critique_b, validation_b = finalize_answer_critique(
    query=QUERY,
    understanding=understanding,
    secondary_evidence=secondary_evidence,
    context_evidence=context_evidence,
    answer=draft_answer,
    contract=contract,
    validation=validation,
    synthesizer=mock_synthesizer,
    parameters=critic_enabled_parameters,
    critic_json_requester=forced_regeneration_critic,
)

print("(b) critic.enabled:", critique_b.enabled, "| source:", critique_b.source)
print("(b) critique.regenerated:", critique_b.regenerated)
print("(b) critic was called", regeneration_critic_calls["count"], "time(s) -- expect exactly 2 (initial + recheck)")
show_json(critique_b)
[(b.text, [c.support_status for c in b.citations]) for b in answer_b.bullets]

(b) critic.enabled: True | source: llm
(b) critique.regenerated: True
(b) critic was called 2 time(s) -- expect exactly 2 (initial + recheck)


<IPython.core.display.JSON object>

[('Regenerated bullet citing the same allowed source.', ['accurate'])]

## 13. Full service call — `QueryAnswerService.answer()`

Ties every step above together exactly as `POST /query-answer` does. Gated by `RUN_LIVE_ANSWER_SYNTHESIS` since it calls the real synthesizer (and the real critic, if `enable_answer_critic` is on).

In [15]:
if RUN_LIVE_ANSWER_SYNTHESIS:
    full_response = query_answer_service.answer(
        QUERY,
        openfda_limit=parameters.default_openfda_limit,
    )
    print("contract.coverage_level:", full_response.contract.coverage_level)
    print("validation.passed:", full_response.validation.passed)
    print("critique.enabled:", full_response.critique.enabled)
    show_json(full_response.critique)
else:
    print("Skipped: would call live answer synthesis. Set RUN_LIVE_ANSWER_SYNTHESIS = True in section 1.")

contract.coverage_level: direct
validation.passed: True
critique.enabled: True


<IPython.core.display.JSON object>

## 14. Gaps and things worth a second look

Observations from tracing this pipeline end to end — not blockers, but worth deciding on deliberately:

1. **[notebooks/03_backend_query_answer_walkthrough.ipynb](03_backend_query_answer_walkthrough.ipynb) is now stale.** It imports and calls `add_coverage_limitations` from `src/query_answer/coverage.py`, which D1 removed in favor of `contract.py` + `validation.py`. Running notebook 03 today raises an `ImportError` at the import cell. It needs an update (or should be retired) so it doesn't mislead anyone about the current pipeline shape.
2. **`critic_max_regenerations` doesn't behave like a count.** `finalize_answer_critique` performs at most one regeneration via a single `if`, not a loop bounded by the parameter's value — so `critic_max_regenerations=2` behaves identically to `=1`; the parameter is really an on/off gate (`> 0`) today. Section 12(b) above demonstrates the actual behavior: the critic is called exactly twice (initial + one recheck) regardless of how high the parameter is set. Either the parameter should be renamed to reflect that it's a boolean-ish gate, or the orchestrator should actually loop up to that count. (Unaffected by D2e — this is about the regeneration *loop*, not what the critic judges.)
3. **The critic's own verdicts are trusted at face value.** D2e made the critic the *only* source of `support_status`, judged against the real cited text instead of a structural section-category match — a real improvement (section 11's live run, and the notebook history below, both show it catching things the old floor structurally couldn't). But nothing downstream re-checks the critic's own reading of `cited_text` programmatically. A critic that misreads or hallucinates confidence about what the label text says has no backstop; D2e trades a weak-but-cheap structural check for a stronger-but-unverified semantic one, not for a verified one.
4. **`AnswerCritique.citations` can be a strict subset of all citations actually in the answer.** If the critic skips a citation, or returns an out-of-range/invalid `bullet_index`/`citation_index`, `_parse_critique` silently drops it (see `MAX_CRITIC_CITATIONS`/`MAX_CRITIC_BULLETS` in section 11, and `critic._parse_critique`'s bounds checks). Unlike the old deterministic floor, there is no fallback that fills in the rest — an unaddressed citation simply keeps `support_status=None`, which renders as *no badge* in the UI. That's an intentional, honest degradation (better than a misleading guess), but it does mean a critic that silently drops citations produces a quieter failure than a wrong-but-present badge would.
5. **Only claims tied to an actual citation get checked against the response.** The critic's second judgment axis — does `response` correctly reflect `cited_text` — only runs for citations that exist. A claim that the model writes directly into the free-text `response` with no citation at all (already a `validate_and_enforce`/D2d concern for *bullets*, but `response` is single free text, not a bullet list) isn't covered by this check at all. `critique.global_findings` is the only backstop there, and it's advisory — nothing enforces it the way `validate_and_enforce` enforces `must_caveat`s.
6. **D2e's accepted cost: every live query is now at least two LLM calls.** Synthesis, then the critic, then a regeneration synthesis + recheck pair if `needs_regeneration` fires. This was an explicit, deliberate trade-off (better faithfulness signal vs. latency/cost), not an oversight — flagging it here so it isn't rediscovered as a surprise later.

### Update — D2b resolved the cross-intent leakage this notebook surfaced

Running this notebook (and the web app) against *"Can I take ibuprofen for my migraine if I'm allergic to aspirin?"* surfaced a real bug: the section 10 output above used to show the interaction-gap bullet (the one stating no `drug_interactions` text was retrieved) flipping between `strong` and `limited` across runs, purely depending on which label section the LLM happened to cite for it. The cause was exactly what item 4 above touches on, but more severe: `deterministic_support_status` pooled `required_sections` across *every* addressed intent into one flat set (`critic._addressed_intent_sections`), so a citation that genuinely backed `allergy_context_check` (e.g. `warnings`) could spuriously "support" an unrelated `interaction_check` bullet just because both intents happened to share a section name.

**Fix (D2b):** each bullet now carries a `topic` (`EvidenceBullet.topic`), parsed from synthesis output and whitelisted against the contract's own topics — the model can select an existing topic but never invent one, same pattern as citation whitelisting. `deterministic_support_status` scopes its section check to that topic's own `required_sections` when a match exists (`critic._topic_required_sections`, exercised in section 10's inspection cell above), falling back to the legacy pooled check only for untagged bullets. `build_answer_contract` also now uses each intent's actually-matched section(s) instead of the static `INTENT_REQUIRED_SECTIONS` allow-list. Re-running the live cells above against the same query now consistently scores that bullet `none`/`limited` regardless of which section gets cited, while the allergy bullets citing `warnings`/`contraindications` correctly remain `strong`.

This doesn't change item 4's underlying point — the floor is still purely structural (it never reads label text); the LLM critic remains the only place that could catch a citation that's structurally valid but semantically unsupportive. D2b closes the specific cross-intent false-positive, not the broader "no semantic re-check" gap.

### Update — D2c fixed an evidence-packet truncation bug this notebook also surfaced

Even after D2b, the same query kept producing a bullet saying the packet didn't include any `drug_interactions` text for ibuprofen — despite the contract (section 6 above) marking `interaction_check` "addressed." Digging into section 7's evidence packet showed why: `label_section_payloads()` walked sections in a fixed priority order (`boxed_warning`, `contraindications`, `warnings`, `drug_interactions`, ...) with a flat 16-entry cap per drug, no per-section sub-limit, and no deduplication. Ibuprofen alone had 2 `boxed_warning` + 2 `contraindications` + 13 `warnings` entries (10 of which were near-identical "allergy alert" boilerplate repeated across different manufacturer label records) = 17 non-empty entries — already past the cap before `drug_interactions` (2 real entries) was ever reached. The deterministic coverage layer (`intent_evidence_status`, section 5) correctly found that evidence on the *untruncated* dossier data, so the contract promised it — but the LLM never actually saw it, because evidence-packet construction truncated independently with no awareness of what the contract relied on.

**Fix (D2c):** `label_section_payloads()` now deduplicates near-identical boilerplate per section before counting it against any cap, caps each section to `MAX_SECTION_ENTRIES` so one bloated section can't crowd out the rest, and guarantees any section the contract relies on (`required_label_sections(contract)`, now shared between `contract.py` and `critic.py` instead of duplicated) survives the overall cap — evicting from the lowest-priority tail if it was already full. Section 7 above now shows `drug_interactions` present for both ibuprofen and aspirin, and a live re-run of this query no longer produces the false "no drug_interactions text" bullet; it now cites the real retrieved aspirin/ibuprofen interaction text instead.

Like D2b, this doesn't replace the still-open structural-only nature of the deterministic floor (item 4) — it fixes a retrieval/packet-construction bug that was starving the floor and the LLM of evidence that was always there.

### Update — D2e removed the deterministic floor this notebook was built around

Sections 10–12 above originally exercised `apply_deterministic_statuses` — the always-on "support status floor" from D2 (topic-scoped since D2b, with its unresolved-gap check further scoped in D2d). The floor's question was purely structural: does this bullet's cited section *category* (e.g. `drug_interactions`) belong to a contract-confirmed intent for the topic the bullet is about? It never read the cited label text or the final response. Item 4 in the original version of section 14 named the consequence directly: *"A critic that hallucinates confidence has no backstop"* — but the more basic problem, raised in conversation while reviewing this exact notebook, was that the floor itself wasn't a real faithfulness check at all. Because the same model call that writes a bullet also picks which section to cite for it, the floor's section-category match was almost always trivially satisfied — it produced a badge that *looked* like a quality signal without actually being one.

**Decision (D2e):** delete the floor outright (`deterministic_support_status`, `apply_deterministic_statuses`, `_topic_required_sections`, `_addressed_intent_sections`, `_has_unresolved_gap`, and the bullet-level `topic` field that only existed to feed it) rather than patch it further, and make the LLM critic — turned **on by default** — the only source of a support-status badge. The critic was also re-scoped from per-bullet to per-*citation* (a bullet can carry more than one), against a new two-axis taxonomy that actually reads the real cited label text and the final response (see the intro cell for the five tiers). A real multi-citation query tried while building D2e (not reproduced live in this notebook, to keep it free of paid calls by default) showed exactly the kind of thing only a real read of `cited_text` vs. `response` could catch: a bullet correctly cited a label's GI-bleeding warning, but the final response text never actually mentioned GI bleeding — the citation faithfully represented its source, yet the response didn't use it, so the critic correctly marked it `not_reflected`. The old floor would have scored that bullet `strong` purely because `boxed_warning`/`warnings` was a section the contract had confirmed as addressed.

This is a real trade-off, not a strict improvement: when the critic can't run (no LLM configured, or `enable_answer_critic` explicitly disabled — e.g. demo mode), citations now carry no `support_status` and no badge renders at all, where the old floor would have shown *something* (structurally derived, but always present). Sections 12(a) and 13 above demonstrate that honestly-empty state. See the rewritten section 14 for what D2e leaves open.